# MapReduce bằng tay: doanh thu theo quốc gia và từ hay gặp trong tên sản phẩm

**Bối cảnh.** Đội data platform đề xuất chuyển báo cáo doanh thu sang Spark "vì dữ liệu sẽ lớn". Head of Analytics
hỏi bạn: *"Spark làm gì khác pandas? Kết quả có giống nhau không, và với dữ liệu hiện tại có nhanh hơn không?"* Cách
tốt nhất để trả lời là tự làm một lần mô hình **MapReduce** mà Spark và Hadoop dựa vào, trên chính dữ liệu bán hàng.

Dữ liệu: bộ `online-retail-ii`, sheet `Year 2010-2011`, dòng hợp lệ theo định nghĩa dùng chung của track (bỏ hoá đơn
bắt đầu bằng `C`, bỏ dòng `Quantity <= 0` hoặc `Price <= 0`; doanh thu gộp = `Quantity * Price`). Hai câu hỏi:

1. Doanh thu theo quốc gia (`Country`).
2. Từ nào hay gặp nhất trong tên sản phẩm (`Description`) của các dòng bán: gợi ý cho đội merchandising về chủ đề
   sản phẩm chủ lực.

Bạn viết map, reduce và một bộ chạy song song bằng `multiprocessing`, rồi so với `groupby` của pandas: kết quả và
thời gian.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật, chạy, đo thời gian; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/14-ml-bigdata-for-analysts/exercises/basic/03-mapreduce-revenue -q`. Xong thì làm phần **trắc nghiệm** trong app.

In [ ]:
# explore: đọc sheet 2010-2011; lần đầu mất 20-90 giây, sau đó đọc bản parquet trong thư mục tạm
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd


def read_retail_1011() -> pd.DataFrame:
    xlsx = ensure("online-retail-ii") / "online_retail_II.xlsx"
    stat = xlsx.stat()
    cache = Path(tempfile.gettempdir()) / "da-cache" / f"retail-1011-{stat.st_size}-{int(stat.st_mtime)}.parquet"
    if cache.exists():
        return pd.read_parquet(cache)
    df = pd.read_excel(xlsx, sheet_name="Year 2010-2011")
    df["Invoice"] = df["Invoice"].astype(str)
    df["StockCode"] = df["StockCode"].astype(str)
    df["Description"] = df["Description"].astype("string")
    cache.parent.mkdir(exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
    df.to_parquet(tmp)
    tmp.replace(cache)
    return df


tx = read_retail_1011()
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())

In [ ]:
import multiprocessing as mp
import re
from collections import Counter

import numpy as np
import pandas as pd

## 1. Dòng hợp lệ và chia khối

`valid_lines(tx) -> DataFrame`: bản sao chỉ gồm dòng hợp lệ (định nghĩa ở trên; ép `Invoice` sang `str` trước khi
kiểm `C`), thêm cột `revenue = Quantity * Price`. Giữ cả dòng không có `Customer ID` (doanh thu theo nước tính mọi
khách, kể cả khách vãng lai).

`split_chunks(df, n_chunks) -> list[DataFrame]`: chia `df` thành đúng `n_chunks` khối **liên tiếp** theo thứ tự dòng,
cỡ các khối chênh nhau tối đa 1 dòng (như `np.array_split`), nối lại thì ra đúng `df`. `n_chunks < 1` thì `ValueError`.
Đây là việc HDFS làm khi cắt tệp thành block, hoặc Spark khi chia partition.

In [ ]:
def valid_lines(tx: pd.DataFrame) -> pd.DataFrame:
    """Dòng hợp lệ (không hoá đơn C, Quantity > 0, Price > 0) kèm cột revenue."""
    # TODO: mask = ~Invoice.astype(str).str.startswith("C") & (Quantity > 0) & (Price > 0)
    # TODO: trả bản sao có thêm revenue
    raise NotImplementedError


def split_chunks(df: pd.DataFrame, n_chunks: int) -> list[pd.DataFrame]:
    """n_chunks khối liên tiếp, cỡ chênh nhau tối đa 1 dòng."""
    # TODO: kiểm n_chunks >= 1
    # TODO: ranh giới khối: np.linspace(0, len(df), n_chunks + 1) làm tròn, hoặc np.array_split trên vị trí dòng
    raise NotImplementedError

## 2. Map và reduce

Mỗi hàm **map** chỉ nhìn **một khối** và trả kết quả từng phần dạng `dict` (khoá → giá trị); nó không biết các khối khác.

- `map_revenue(chunk) -> dict[str, float]`: doanh thu theo `Country` trên dòng hợp lệ của khối (dùng `valid_lines`).
- `map_words(chunk) -> dict[str, int]`: trên dòng hợp lệ của khối, đếm số lần mỗi từ xuất hiện trong `Description`.
  Từ = chuỗi chữ cái `a-z` liên tiếp sau khi đổi về chữ thường (`re.findall(r"[a-z]+", text.lower())`), nên
  `"SET/3 RED"` cho `set`, `red`. Bỏ qua `Description` trống (NaN). Một dòng có từ lặp hai lần thì đếm hai lần.

`reduce_counts(partials) -> dict`: cộng các dict từng phần theo khoá. Khoá chỉ có ở một vài khối vẫn được giữ.

In [ ]:
def map_revenue(chunk: pd.DataFrame) -> dict[str, float]:
    """Doanh thu theo Country trên dòng hợp lệ của một khối."""
    # TODO: valid_lines(chunk).groupby("Country")["revenue"].sum(), đổi sang dict
    raise NotImplementedError


def map_words(chunk: pd.DataFrame) -> dict[str, int]:
    """Đếm từ (a-z, chữ thường) trong Description của dòng hợp lệ trong một khối."""
    # TODO: Counter; với mỗi Description khác NaN: re.findall(r"[a-z]+", text.lower())
    raise NotImplementedError


def reduce_counts(partials: list[dict]) -> dict:
    """Cộng các dict từng phần theo khoá."""
    # TODO
    raise NotImplementedError

## 3. Chạy song song và so với pandas

`run_mapreduce(df, mapper, n_workers) -> dict`: chia `df` thành `n_workers` khối, chạy `mapper` trên từng khối, rồi
`reduce_counts`. Với `n_workers == 1` chạy tuần tự trong tiến trình hiện tại; với `n_workers > 1` dùng một
`multiprocessing.Pool(n_workers)` (dùng context `"fork"` nếu hệ điều hành có, xem gợi ý trong đề) và `pool.map`.

`pandas_revenue(tx) -> dict[str, float]`: cùng câu hỏi 1 bằng một lần `groupby` trên toàn bộ dữ liệu, để đối chiếu.

In [ ]:
def run_mapreduce(df: pd.DataFrame, mapper, n_workers: int = 2) -> dict:
    """Chia khối, map song song (multiprocessing), reduce."""
    # TODO: chunks = split_chunks(df, n_workers)
    # TODO: n_workers == 1: [mapper(c) for c in chunks]
    # TODO: ngược lại: ctx = mp.get_context("fork") nếu "fork" có trong mp.get_all_start_methods()
    #       with ctx.Pool(n_workers) as pool: partials = pool.map(mapper, chunks)
    raise NotImplementedError


def pandas_revenue(tx: pd.DataFrame) -> dict[str, float]:
    """Doanh thu theo Country bằng một lần groupby."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kết quả có giống nhau không?
by_pandas = pandas_revenue(tx)
by_mapreduce = run_mapreduce(tx, map_revenue, n_workers=4)
diff = max(abs(by_pandas[c] - by_mapreduce[c]) for c in by_pandas)
print(f"{len(by_pandas)} quốc gia; tổng {sum(by_pandas.values()):,.2f} GBP; lệch lớn nhất giữa hai cách: {diff:.2e} GBP")
top = pd.Series(by_mapreduce).sort_values(ascending=False)
print((top.head(8) / top.sum()).round(4))

In [ ]:
# explore: từ hay gặp nhất trong tên sản phẩm của dòng bán
words = pd.Series(run_mapreduce(tx, map_words, n_workers=4)).sort_values(ascending=False)
print(f"{len(words):,} từ khác nhau")
print(words.head(15))

In [ ]:
# explore: nhanh hơn không? (số đo thay đổi theo máy; lần chạy đầu của Pool có chi phí khởi tạo)
import time

for label, fn in [("pandas groupby", lambda: pandas_revenue(tx)),
                  ("mapreduce, 1 worker", lambda: run_mapreduce(tx, map_revenue, 1)),
                  ("mapreduce, 2 worker", lambda: run_mapreduce(tx, map_revenue, 2)),
                  ("mapreduce, 4 worker", lambda: run_mapreduce(tx, map_revenue, 4))]:
    start = time.perf_counter()
    fn()
    print(f"{label:<22} {time.perf_counter() - start:6.2f} s")

Viết ra (cho chính bạn, trước khi làm trắc nghiệm):

1. Hai cách cho cùng kết quả không? Nếu lệch, lệch ở chữ số thứ mấy, và vì sao?
2. Với 530 nghìn dòng, chạy song song có nhanh hơn `groupby` không? Phần thời gian đi vào đâu?
3. Nếu thay doanh thu bằng **giá trị trung bình mỗi dòng** theo nước, map phải trả gì để reduce vẫn đúng?

Xong thì mở app, làm phần **trắc nghiệm** của bài.